# 02 · Feature engineering and leakage review

In [1]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Image
from src import config
pd.set_option('display.width', 160); sns.set_theme(style='whitegrid')
from src import preprocessing as pp
from src.ingestion import load_raw_excel
df = pd.read_csv(config.DATA_PROCESSED / 'customers_processed.csv')   # cleaned + engineered (written by ingestion)
display(pp.feature_dictionary())

,feature,type,description
0,tenure_months,observed,source column (cleaned)
1,monthly_charges,observed,source column (cleaned)
2,total_charges,observed,source column (cleaned)
3,senior_citizen,observed,source column (cleaned)
4,partner,observed,source column (cleaned)
5,dependents,observed,source column (cleaned)
6,phone_service,observed,source column (cleaned)
7,multiple_lines,observed,source column (cleaned)
8,online_security,observed,source column (cleaned)
9,online_backup,observed,source column (cleaned)


## Leakage assessment of IBM's precomputed fields

In [2]:
import json; a = pp.assess_cltv_leakage(pp.clean_raw(load_raw_excel())); print(json.dumps(a, indent=2))

{
  "corr_cltv_churn": -0.12746310171744962,
  "corr_cltv_churn_score": -0.07978218909612224,
  "corr_churn_score_churn": 0.6648970311816242,
  "corr_cltv_tenure": 0.39640568032795814,
  "corr_cltv_monthly_charges": 0.09869320961239764,
  "decision": "EXCLUDED: IBM's CLTV is a precomputed, undocumented-methodology field of unknown timing (it may embed churn/survival information). It is kept only as a reference column, never as a model input; this project derives its own transparent CLV (src/clv.py)."
}


`Churn Score` correlates strongly with the target (it is a model output); `Churn Reason` exists only for churners. Both, plus `CLTV` (opaque methodology), are excluded from all models. This project derives its own CLV.

In [3]:
X = pp.model_frame(df); print(X.shape, 'NaNs:', int(X.isna().sum().sum()))
assert not set(X.columns) & set(pp.LEAKAGE_COLUMNS)
Xt = pp.build_preprocessor().fit_transform(X); print('encoded matrix', Xt.shape)

(7043, 34) NaNs: 0
encoded matrix (7043, 48)


In [4]:
display(df.groupby('churn')[pp.ENGINEERED_NUMERIC].mean().T.round(3))

churn,0,1
avg_monthly_spend,61.270,74.433
charge_ratio,1.002,1.003
contract_months,11.100,2.568
is_month_to_month,0.429,0.886
service_count,4.173,4.072
addon_count,2.135,1.768
has_internet,0.727,0.940
has_protection_bundle,0.193,0.053
auto_pay,0.498,0.262
electronic_check,0.250,0.573
